# 02 · Pipeline: de MASTER a FS (variables reutilizables)

**FS** guarda variables por **cliente y fecha de corte**. Es una capa didáctica, no una plataforma de *feature store*.

> **Regla de oro:** cada variable se calcula **solo con datos anteriores al corte**. Así el modelo nunca ve el futuro (*fuga de información*).

Necesita MASTER construido (cuaderno 01).

In [ ]:
# Parámetros. None usa las rutas por defecto del proyecto (o las variables de entorno CURSO_DB y CURSO_REPORTES).
RUTA_DB = None
RUTA_REPORTES = None
CORTES = {
    'train': ['2010-06-01', '2010-07-01', '2010-08-01', '2010-09-01', '2010-10-01', '2010-11-01', '2010-12-01'],
    'valid': ['2011-03-01'],
    'test': ['2011-06-01', '2011-09-01'],
}

In [ ]:
import os, sqlite3, sys
from pathlib import Path

import pandas as pd

pd.set_option('display.width', 150, 'display.max_columns', 30)

# Carpeta del proyecto: la que contiene «sql» y «pyproject.toml».
RAIZ = Path.cwd().resolve()
while not ((RAIZ / 'sql').exists() and (RAIZ / 'pyproject.toml').exists()) and RAIZ != RAIZ.parent:
    RAIZ = RAIZ.parent
SQL = RAIZ / 'sql'
DB = Path(RUTA_DB or os.environ.get('CURSO_DB') or RAIZ / 'data' / 'processed' / 'curso.db')
DB.parent.mkdir(parents=True, exist_ok=True)

con = sqlite3.connect(DB)
con.execute('PRAGMA foreign_keys = ON')


def ejecutar_script(archivo):
    """Ejecuta un archivo .sql de la carpeta sql/ (puede tener varias sentencias)."""
    con.executescript((SQL / archivo).read_text(encoding='utf-8'))
    con.commit()


def escalar(consulta, parametros=None):
    fila = con.execute(consulta, parametros or {}).fetchone()
    return None if fila is None else fila[0]


print('Proyecto:', RAIZ)
print('Base de datos:', DB)
REPORTES = Path(RUTA_REPORTES or os.environ.get('CURSO_REPORTES') or RAIZ / 'reports')
(REPORTES / 'figures').mkdir(parents=True, exist_ok=True)

## 1. El catálogo de variables y la tabla destino
`sql/fs/01_crear_fs.sql` crea `fs_catalogo` (nombre, descripción, fórmula y unidad de cada variable) y `fs_cliente_corte`.

In [ ]:
ejecutar_script('fs/01_crear_fs.sql')
pd.read_sql('SELECT feature, descripcion, formula, unidad FROM fs_catalogo', con)

## 2. Poblar FS: una vez por fecha de corte
`sql/fs/02_poblar_fs.sql` es **una sola sentencia** con el parámetro `:corte`. Léela: todos sus filtros dicen `fecha_factura < :corte`.

In [ ]:
sql_fs = (SQL / 'fs' / '02_poblar_fs.sql').read_text(encoding='utf-8')
todos_los_cortes = [c for lista in CORTES.values() for c in lista]
for corte in todos_los_cortes:
    con.execute(sql_fs, {'corte': corte})
    con.commit()
    print(f'corte {corte}: {escalar("SELECT COUNT(*) FROM fs_cliente_corte WHERE fecha_corte = :c", {"c": corte}):,} clientes')

In [ ]:
pd.read_sql('SELECT * FROM fs_cliente_corte ORDER BY fecha_corte, cliente_id LIMIT 6', con)

In [ ]:
pd.read_sql('SELECT fecha_corte, COUNT(*) AS clientes, ROUND(AVG(recencia_dias),1) AS recencia_media, ROUND(AVG(frecuencia),1) AS frecuencia_media, ROUND(AVG(monto_total),0) AS monto_medio FROM fs_cliente_corte GROUP BY fecha_corte ORDER BY fecha_corte', con)

## 3. Comprobar la regla de oro con un cliente
Recalculamos **a mano**, con pandas, la recencia y la frecuencia de un cliente usando solo lo anterior al corte, y comparamos con FS.

In [ ]:
corte = CORTES['train'][-1]
cliente = int(escalar('SELECT cliente_id FROM fs_cliente_corte WHERE fecha_corte = :c ORDER BY frecuencia DESC LIMIT 1', {'c': corte}))
ventas = pd.read_sql("""
    SELECT f.factura_id, f.fecha_factura FROM master_lineas l JOIN master_facturas f ON f.factura_id = l.factura_id
    WHERE l.es_venta_valida = 1 AND f.cliente_id = :c GROUP BY f.factura_id""", con, params={'c': cliente})
ventas['fecha'] = pd.to_datetime(ventas['fecha_factura'])
antes = ventas[ventas['fecha'] < pd.Timestamp(corte)]
despues = ventas[ventas['fecha'] >= pd.Timestamp(corte)]
fs = pd.read_sql('SELECT recencia_dias, frecuencia FROM fs_cliente_corte WHERE cliente_id = :c AND fecha_corte = :f', con, params={'c': cliente, 'f': corte}).iloc[0]
recencia_a_mano, frecuencia_a_mano = (pd.Timestamp(corte) - antes['fecha'].max()).days, len(antes)
print(f'Cliente {cliente} · corte {corte}')
print('FS dice:       recencia', int(fs.recencia_dias), '| frecuencia', int(fs.frecuencia))
print('A mano:        recencia', recencia_a_mano, '| frecuencia', frecuencia_a_mano)
print('Facturas del cliente DESPUÉS del corte (el modelo NO debe verlas):', len(despues))
assert (int(fs.recencia_dias), int(fs.frecuencia)) == (recencia_a_mano, frecuencia_a_mano)

## 4. La prueba definitiva contra la fuga
`tests/test_fs_y_ml.py::test_fs_no_usa_informacion_del_futuro` borra **todo** lo posterior a un corte, recalcula FS y comprueba que las variables no cambian.
Ejecútala en una terminal: `pytest tests -k futuro`.

## 5. Mirar las variables

In [ ]:
import matplotlib
matplotlib.use('Agg') if os.environ.get('CURSO_SIN_PANTALLA') else None
import matplotlib.pyplot as plt

df = pd.read_sql('SELECT * FROM fs_cliente_corte WHERE fecha_corte = :c', con, params={'c': corte})
fig, ax = plt.subplots(1, 3, figsize=(13, 3.2))
df['recencia_dias'].plot.hist(bins=30, ax=ax[0], title='recencia_dias')
df['frecuencia'].clip(upper=30).plot.hist(bins=30, ax=ax[1], title='frecuencia (recortada a 30)')
df['monto_total'].clip(upper=df['monto_total'].quantile(0.95)).plot.hist(bins=30, ax=ax[2], title='monto_total (hasta el percentil 95)')
plt.tight_layout()
fig.savefig(REPORTES / 'figures' / 'fs_distribuciones.png', dpi=110)
plt.show()

In [ ]:
con.close()
print('Listo: FS construida en', DB)

## Para practicar (tarea de S1)
**Adapta una variable.** Añade una variable nueva a FS, por ejemplo `dias_desde_ultima_cancelacion` o `monto_ultima_factura`:
1. Añade la columna en `sql/fs/01_crear_fs.sql` y su cálculo en `02_poblar_fs.sql`, usando **solo** `fecha_factura < :corte`.
2. Descríbela en `fs_catalogo`.
3. Comprueba con la prueba de «sin información del futuro» que no hay fuga.